ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

# Make plots look clean
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv')

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df.drop('CUST_ID', axis=1, inplace=True)

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df.fillna(df.mean(), inplace=True)

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(bins=30, figsize=(20, 15), color='steelblue', edgecolor='white')
plt.suptitle('Distribution of All Numerical Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(16, 12))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            annot_kws={'size': 8})
plt.title('Correlation Heatmap of Credit Card Features', fontsize=14)
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.3, color='steelblue', edgecolors='none')
plt.xlabel('Balance')
plt.ylabel('Purchases')
plt.title('Balance vs Purchases')
plt.tight_layout()
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.3, color='coral', edgecolors='none')
plt.xlabel('Balance')
plt.ylabel('Cash Advance')
plt.title('Balance vs Cash Advance')
plt.tight_layout()
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop('Cluster', axis=1, errors='ignore'))
print("X_scaled shape:", X_scaled.shape)
print("Mean of first feature after scaling:", X_scaled[:, 0].mean().round(5))

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia_values.append(km.inertia_)

print("K → Inertia")
for k, inertia in zip(range(1, 11), inertia_values):
    print(f"  K={k}: {inertia:.1f}")

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, 11), inertia_values, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Chosen K=3')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
plt.title('Elbow Method for Optimal K', fontsize=14)
plt.xticks(range(1, 11))
plt.legend()
plt.tight_layout()
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), silhouette_scores, marker='s', color='darkorange',
         linewidth=2, markersize=8)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Chosen K=3')
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Scores for K = 2 to 10', fontsize=14)
plt.xticks(range(2, 11))
plt.legend()
plt.tight_layout()
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_table = pd.DataFrame({
    'K': range(2, 11),
    'Silhouette Score': [round(s, 4) for s in silhouette_scores]
})
silhouette_table = silhouette_table.set_index('K')
silhouette_table['Best?'] = silhouette_table['Silhouette Score'] == silhouette_table['Silhouette Score'].max()
print(silhouette_table.to_string())

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
# K=3 is chosen:
#   - The elbow curve shows a clear bend at K=3 (inertia drop slows significantly after K=3).
#   - K=3 achieves the highest silhouette score (≈0.2506), better than K=2 (0.210) and K=4 (0.198).
#   - Three segments also make strong business sense: spenders, cash-advance users, and average users.

km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
km_final.fit(X_scaled)
print("Final model trained with K=3")
print(f"Inertia: {km_final.inertia_:.1f}")

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = km_final.labels_
print("Cluster column added successfully.")

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean().round(2)
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
cluster_sizes = df['Cluster'].value_counts().sort_index()
print("Customers per cluster:")
print(cluster_sizes)

# Bar chart of cluster sizes
plt.figure(figsize=(6, 4))
cluster_sizes.plot(kind='bar', color=['steelblue', 'darkorange', 'seagreen'], edgecolor='white')
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)
plt.title('Number of Customers per Cluster', fontsize=13)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f"Variance explained by PC1: {explained[0]*100:.1f}%")
print(f"Variance explained by PC2: {explained[1]*100:.1f}%")
print(f"Total variance captured: {sum(explained)*100:.1f}%")

# Plot
colors = {0: 'steelblue', 1: 'darkorange', 2: 'seagreen'}
labels_map = {0: 'Cluster 0: Cash-Advance Users', 
              1: 'Cluster 1: High Spenders',
              2: 'Cluster 2: Average/Low Activity'}

plt.figure(figsize=(10, 7))
for cluster in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors[cluster], label=labels_map[cluster],
                alpha=0.4, s=20, edgecolors='none')

plt.xlabel(f'Principal Component 1 ({explained[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'Principal Component 2 ({explained[1]*100:.1f}% variance)', fontsize=12)
plt.title('K-Means Clusters Visualized with PCA (2 Components)', fontsize=14)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

---

**1. Why is this an unsupervised learning problem?**

This is an unsupervised learning problem because the dataset does not contain a predefined target variable or label indicating which group a customer belongs to. There is no "correct answer" to train against. Instead, the algorithm discovers hidden patterns and natural groupings in the data on its own, without any guidance from labeled examples. We do not tell the model what a "good" or "bad" customer looks like — it finds the structure by itself.

---

**2. Why did we remove the `CUST_ID` column?**

`CUST_ID` is a unique customer identifier — it is simply an ID number that was assigned to each row for tracking purposes. It carries no behavioral information about the customer. Including it in clustering would cause the algorithm to treat customer ID numbers as a meaningful feature and calculate distances based on arbitrary identifiers, which would distort and corrupt the clustering results. Therefore, it must be removed before any analysis or modeling.

---

**3. Which columns had missing values?**

Two columns contained missing values:
- `CREDIT_LIMIT`: 1 missing value
- `MINIMUM_PAYMENTS`: 313 missing values

All other columns were complete with no missing entries.

---

**4. How did you handle the missing values?**

We used **mean imputation**: each missing value was replaced with the mean (average) of that column calculated from all available non-missing values. This is a simple and widely-used technique that preserves the overall statistical distribution of the column and does not reduce the number of rows. It is appropriate here because the missing values are relatively few compared to the 8,950 total rows, and dropping rows would unnecessarily reduce the dataset size.

---

**5. Why is scaling important before applying K-Means?**

K-Means calculates the **Euclidean distance** between data points and cluster centroids to assign each point to the nearest cluster. If the features have very different numerical ranges — for example, `CREDIT_LIMIT` can exceed 30,000 while `PURCHASES_FREQUENCY` is between 0 and 1 — then the large-scale features will dominate the distance calculation and make the small-scale features virtually irrelevant, even if they are equally or more important for segmentation. StandardScaler transforms each feature to have a mean of 0 and a standard deviation of 1, giving all features equal weight and allowing K-Means to make fair, balanced comparisons.

---

**6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.**

We chose **K = 3**.

*Elbow Method:* The inertia drops sharply from K=1 (152,150) to K=2 (127,785) and again to K=3 (111,987). After K=3, the curve begins to flatten noticeably — going from K=3 to K=4 reduces inertia by only about 12,913 compared to the much larger drop of 15,800 from K=2 to K=3. The "elbow" — the point where adding more clusters yields diminishing returns — is clearly visible at K=3.

*Silhouette Score:* K=3 achieved the highest silhouette score of **0.2506**, outperforming K=2 (0.210), K=4 (0.198), and all higher values. This confirms that K=3 produces the most internally cohesive and well-separated clusters among all options tested.

*Business Rationale:* Three clusters also make practical business sense, producing three distinct and interpretable customer segments: cash-advance-heavy users, high-purchasing active spenders, and average low-activity customers. This is directly actionable for marketing strategy.

---

**7. Based on the cluster summary table, describe each customer segment in your own words.**

| Cluster | Name | Description |
|---------|------|-------------|
| **Cluster 0** | Cash-Advance Users | These customers carry high balances (avg ≈ $3,989) and take out large cash advances (avg ≈ $3,870). They make very few purchases (avg ≈ $385) and almost never pay their full balance (full payment rate ≈ 3%). They use their card primarily as a borrowing tool, not a spending tool. They carry significant financial risk. |
| **Cluster 1** | High-Value Active Spenders | These customers make very high purchases (avg ≈ $4,269) with a high purchase frequency (0.95 — meaning they buy almost every month). They have the highest credit limits (avg ≈ $7,734) and the highest payment amounts (avg ≈ $4,151). They pay 30% of their balance in full. They are the most financially active and valuable customers. |
| **Cluster 2** | Average / Low-Activity Users | The largest group (6,119 customers), these are everyday users with moderate balances (avg ≈ $800) and moderate purchases (avg ≈ $506). They have lower credit limits (avg ≈ $3,270) and lower payments. They are not heavy users but represent the typical, stable customer base. |

---

**8. Which cluster may represent high-value customers?**

**Cluster 1 (High-Value Active Spenders)** represents the high-value customers. They have the highest average purchases ($4,268.52), the highest credit limits ($7,733.97), and the highest payment amounts ($4,151.28). Their purchase frequency of 0.95 means they use their card almost every billing cycle. They also have a 30% full-payment rate, indicating financial responsibility. From a business perspective, these customers generate the most transaction revenue and represent the most profitable segment.

---

**9. Which cluster may represent customers who rely more on cash advance?**

**Cluster 0** clearly represents cash-advance-reliant customers. Their average cash advance is $3,869.86 — nearly ten times higher than Cluster 1 ($458.42) and Cluster 2 ($329.87). Their cash advance frequency is 0.45, meaning they take advances nearly every other month. At the same time, their purchases are very low ($384.53) and their full-payment rate is only 3%, suggesting they are using their credit card primarily to borrow cash rather than to make purchases. This is typically associated with financial stress.

---

**10. How can a company use these clusters for marketing strategy?**

**Strategy for Cluster 1 – High-Value Spenders (Reward & Retain):**
These are the most profitable customers. The company should focus on retention and deepening engagement. Offer premium rewards programs (e.g., travel miles, cashback on purchases), raise their credit limits proactively, and provide exclusive membership perks. Targeted campaigns could offer bonus rewards for reaching monthly spending thresholds, encouraging even higher purchase volumes.

**Strategy for Cluster 0 – Cash-Advance Users (Financial Support & Risk Management):**
These customers are financially stressed and rely on their card for liquidity. The company has two objectives: (1) manage credit risk by monitoring these accounts for default signals, and (2) offer financial support products such as lower cash-advance interest rates, balance transfer options, or financial wellness programs. Converting some of these customers to installment loans may reduce risk while keeping them engaged with the brand.

**Strategy for Cluster 2 – Average/Low-Activity Users (Activation & Upsell):**
This is the largest segment but the least engaged. The company should run activation campaigns — such as limited-time double-point promotions, introductory cashback offers on everyday spending categories (groceries, fuel), or personalized spending challenges — to encourage more frequent use. Some of these customers may be candidates for credit limit increases, which typically correlate with higher spending. Moving these customers toward Cluster 1 behavior is the ultimate goal.